# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 


## **1. Basic Categorical Encoders**

### **1.1. Ordinal Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_01.jpg?v=1783952509" width="250">



>* Convert ordered categories into integer codes
>* Preserve meaningful rank and relative position

>* Models may treat codes as quantities
>* Unordered categories can create false patterns

>* Use only with justified category order
>* Check spacing assumptions and alternatives



In [ ]:
#@title Python Code - Ordinal Encoding

# Ordinal encoding maps ordered categories to numbers.
# Use it only when category order is meaningful.
# This example uses plain Python and pandas.

import pandas as pd

# Create a tiny dataset with ordered satisfaction levels.
data = {
    "customer": ["A", "B", "C", "D", "E"],
    "satisfaction": ["poor", "good", "excellent", "fair", "very good"],
}

# Convert the dictionary into a small DataFrame.
df = pd.DataFrame(data)

# Define the intended order from lowest to highest.
ordered_levels = ["poor", "fair", "good", "very good", "excellent"]

# Build a mapping from category names to integers.
ordinal_map = {level: index for index, level in enumerate(ordered_levels)}

# Apply the mapping to create encoded numeric values.
df["satisfaction_code"] = df["satisfaction"].map(ordinal_map)

# Check that every category was successfully encoded.
missing_codes = df["satisfaction_code"].isna().sum()

# Print compact results for easy inspection.
print("Ordinal category order:", ordered_levels)
print("Missing encoded values:", int(missing_codes))
print(df.to_string(index=False))



### **1.2. One Hot Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_02.jpg?v=1783952512" width="250">



>* Creates indicators for unordered categories
>* Avoids misleading numeric rankings in models

>* Cleanly represents unordered nominal categories
>* Learns separate effects for each category

>* Best for manageable nominal categories
>* Watch indicator growth and redundancy



In [ ]:
#@title Python Code - One Hot Encoding

# Demonstrate one hot encoding basics.
# Use pandas without machine learning libraries.
# Keep output short and readable.

import pandas as pd

# Create a tiny categorical dataset.
homes = pd.DataFrame(
    {
        "home_id": [101, 102, 103, 104],
        "heating": ["gas", "electric", "gas", "solar"],
        "size_sqft": [850, 920, 780, 1100],
    }
)

# Encode the nominal heating feature.
encoded = pd.get_dummies(
    homes,
    columns=["heating"],
    dtype=int,
)

# Validate the expected encoded width.
expected_columns = 2 + homes["heating"].nunique()
actual_columns = encoded.shape[1]
assert actual_columns == expected_columns

# Show the original categories.
print("Original heating values:")
print(homes["heating"].tolist())

# Show compact encoded results.
print("\nOne hot encoded columns:")
print(encoded.columns.tolist())

# Show each row after encoding.
print("\nEncoded table preview:")
print(encoded.to_string(index=False))



### **1.3. Pandas Categories**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_03.jpg?v=1783952514" width="250">



>* Mark repeated labels as categorical values
>* Prepare categories for cleaner model encoding

>* Categories save memory with hidden integer codes
>* Choose ordinal or one-hot encoding deliberately

>* Define valid categories and meaningful order
>* Still encode categories for machine learning



In [ ]:
#@title Python Code - Pandas Categories

# Pandas categories clarify repeated text labels.
# Category order can document real meaning.
# Internal codes are storage, not modeling.

# Import pandas for categorical data handling.
import pandas as pd

# Create a tiny dataset with repeated labels.
data = {
    "size": ["small", "medium", "large", "medium"],
    "channel": ["online", "store", "phone", "online"],
}

# Build a small pandas DataFrame.
df = pd.DataFrame(data)

# Define an ordered categorical size type.
size_type = pd.CategoricalDtype(
    categories=["small", "medium", "large"],
    ordered=True,
)

# Define an unordered categorical channel type.
channel_type = pd.CategoricalDtype(
    categories=["online", "store", "phone"],
    ordered=False,
)

# Convert text columns into pandas categories.
df["size"] = df["size"].astype(size_type)
df["channel"] = df["channel"].astype(channel_type)

# Inspect category metadata without printing everything.
print("Size categories:", list(df["size"].cat.categories))
print("Size ordered:", df["size"].cat.ordered)

# Show compact internal codes for storage.
print("Size internal codes:", df["size"].cat.codes.tolist())
print("Channel internal codes:", df["channel"].cat.codes.tolist())

# Compare memory use before and after conversion.
text_df = pd.DataFrame(data)
text_memory = text_df.memory_usage(deep=True).sum()
category_memory = df.memory_usage(deep=True).sum()

# Print a short memory comparison.
print("Text memory bytes:", int(text_memory))
print("Category memory bytes:", int(category_memory))

# Demonstrate an unexpected category becomes missing.
new_size = pd.Series(["medium", "extra large"], dtype=size_type)
print("Unknown label becomes missing:", new_size.isna().tolist())



## **2. Category Handling**

### **2.1. Handling Unknown Categories**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_01.jpg?v=1783952504" width="250">



>* New categories often appear after training
>* Plan handling to prevent pipeline failures

>* Choose handling based on encoder and meaning
>* Avoid misleading values while keeping pipelines stable

>* Store training categories to prevent leakage
>* Monitor unknowns and retrain when needed



In [ ]:
#@title Python Code - Handling Unknown Categories

# Demonstrate unknown category handling safely.
# Use tiny data for clear output.
# Compare strict and tolerant encoders.

import pandas as pd

# Create training categories seen during fitting.
train = pd.Series(["Canada", "India", "Brazil", "India"])

# Create deployment categories with one unseen value.
new_data = pd.Series(["India", "Kenya", "Brazil"])

# Learn categories only from training data.
known_categories = sorted(train.unique().tolist())

# Build one-hot columns from learned categories.
encoded = pd.get_dummies(new_data)
encoded = encoded.reindex(columns=known_categories, fill_value=0)

# Detect values not learned during training.
unknown_mask = ~new_data.isin(known_categories)
unknown_values = new_data[unknown_mask].tolist()

# Add an explicit unknown indicator column.
encoded["Unknown"] = unknown_mask.astype(int)

# Show compact teaching output only.
print("Known training categories:", known_categories)
print("New incoming values:", new_data.tolist())
print("Unknown values detected:", unknown_values)
print("Safe encoded rows:")
print(encoded.to_string(index=False))



### **2.2. Infrequent Category Grouping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_02.jpg?v=1783952506" width="250">



>* Rare categories create large, sparse encodings
>* Group rare values into an infrequent label

>* Improves model stability and generalization
>* Reduces memorization in high-cardinality features

>* Set rarity thresholds based on context
>* Preserve meaningful rare-category signals



In [ ]:
#@title Python Code - Infrequent Category Grouping

# Demonstrate grouping rare categorical values.
# Keep the dataset intentionally tiny.
# Compare raw and grouped encodings.

import pandas as pd

# Create a small high-cardinality feature.
brands = [
    "Acme", "Acme", "Acme", "Acme",

    "Bravo", "Bravo", "Bravo", "Bravo",
    "Cedar", "Cedar", "Delta", "Echo",

    "Fjord", "Grove", "Halo", "Iris",
]

# Store the feature in a dataframe.
df = pd.DataFrame({"brand": brands})

# Count how often each category appears.
counts = df["brand"].value_counts()

# Keep categories appearing at least three times.
frequent = counts[counts >= 3].index

# Replace rare categories with one shared label.
df["brand_grouped"] = df["brand"].where(
    df["brand"].isin(frequent), "Infrequent"
)

# One-hot encode before and after grouping.
raw_encoded = pd.get_dummies(df["brand"], prefix="brand")
grouped_encoded = pd.get_dummies(df["brand_grouped"], prefix="brand")

# Validate that row counts stayed unchanged.
assert raw_encoded.shape[0] == grouped_encoded.shape[0]

# Print compact teaching results.
print("Original unique categories:", df["brand"].nunique())
print("Grouped unique categories:", df["brand_grouped"].nunique())
print("Raw one-hot columns:", raw_encoded.shape[1])
print("Grouped one-hot columns:", grouped_encoded.shape[1])
print("Frequent categories kept:", list(frequent))
print("Grouped counts:", df["brand_grouped"].value_counts().to_dict())



### **2.3. Drop Options**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_03.jpg?v=1783952508" width="250">



>* Drop one category to avoid redundant indicators
>* Use it as the model’s reference category

>* Choose a meaningful reference category carefully
>* All-zero indicators represent the dropped category

>* Coordinate drops with rare and unknown categories
>* Choose drops based on model type



In [ ]:
#@title Python Code - Drop Options

# Demonstrate one hot drop options.
# Compare full and reference encodings.
# Keep output short and readable.

import pandas as pd

# Create a tiny categorical dataset.
payments = pd.Series(
    ["cash", "card", "transfer", "cash", "card"],
    name="payment_method",
)

# Encode every category as indicators.
full_encoded = pd.get_dummies(
    payments,
    prefix="pay",
    dtype=int,
)

# Drop one category as reference.
dropped_encoded = pd.get_dummies(
    payments,
    prefix="pay",
    drop_first=True,
    dtype=int,
)

# Show the retained column names.
print("Full columns:", list(full_encoded.columns))
print("Dropped columns:", list(dropped_encoded.columns))

# Identify the implicit reference category.
full_categories = set(full_encoded.columns)
kept_categories = set(dropped_encoded.columns)
reference_column = sorted(full_categories - kept_categories)[0]

# Show how the reference appears.
reference_rows = dropped_encoded.sum(axis=1).eq(0)
reference_count = int(reference_rows.sum())
print("Implicit reference:", reference_column)
print("Reference rows encoded all zero:", reference_count)

# Demonstrate an unknown category risk.
new_payment = pd.Series(["crypto"], name="payment_method")
new_encoded = pd.get_dummies(new_payment, prefix="pay", dtype=int)
new_aligned = new_encoded.reindex(columns=dropped_encoded.columns, fill_value=0)

# Unknown can resemble the dropped baseline.
unknown_looks_reference = bool(new_aligned.sum(axis=1).iloc[0] == 0)
print("Unknown looks like reference:", unknown_looks_reference)



## **3. Encoder Pipelines**

### **3.1. Target Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_01.jpg?v=1783952516" width="250">



>* Replace categories with target-based summaries
>* Learn mappings during pipeline preprocessing

>* Prevent leakage with pipelines and cross-validation
>* Smooth rare categories toward overall averages

>* Apply encoders selectively by column type
>* Ensure consistent preprocessing for future data



### **3.2. Sparse Encoder Output**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_02.jpg?v=1783952518" width="250">



>* Sparse encoding saves memory for many categories
>* Output format affects later workflow steps

>* Sparse output saves memory at scale
>* Check downstream tools for sparse compatibility

>* Combine sparse and dense branches carefully
>* Match final matrix to model needs



### **3.3. ColumnTransformer Workflows**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_03.jpg?v=1783952520" width="250">



>* Apply column-specific preprocessing in one workflow
>* Keep transformations consistent across model stages

>* Match encoders to appropriate categorical columns
>* Pipeline steps ensure consistent future transformations

>* Validate full pipelines to prevent leakage
>* Preserve category handling for safer deployment



# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>


In this lecture, you learned to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 

In the next Lecture (Lecture B), we will go over 'Imputation Targets'